In [32]:
import pandas as pd
import ast

In [33]:
# Load the raw dataset
df = pd.read_csv('./data/RAW_recipes.csv')

In [34]:
df.sample(5)

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
11159,avocado chicken mango salad with hazelnut hone...,123077,40,197023,2005-05-21,"['60-minutes-or-less', 'time-to-make', 'course...","[366.3, 43.0, 62.0, 25.0, 15.0, 17.0, 8.0]",15,"['place the chicken breasts in a pan , add the...",the contrasting colours in this salad are appe...,"['chicken breast fillets', 'chicken stock', 'f...",12
65861,crock pot gingerbread pudding cake,334853,135,98812,2008-11-03,"['time-to-make', 'preparation', 'low-protein',...","[493.5, 38.0, 197.0, 20.0, 6.0, 65.0, 22.0]",13,['lightly coat the inside of a 3-1 / 2 or 4-qu...,"this is a cake-on-top, pudding-on-bottom desse...","['nonstick cooking spray', 'gingerbread mix', ...",8
194699,spiced cider tea,472687,5,893970,2012-01-23,"['15-minutes-or-less', 'time-to-make', 'course...","[16.6, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0]",3,"['combine the cider , cinnamon stick , cloves ...",what a delightful drink--like a tea cider! en...,"['apple cider', 'cinnamon stick', 'whole clove...",5
212989,this is howard cosell s sour cream cake robust...,180647,75,207176,2006-08-07,"['celebrity', 'time-to-make', 'course', 'main-...","[883.5, 66.0, 252.0, 34.0, 25.0, 117.0, 38.0]",10,"['preheat oven to 350 degrees', 'cream butter ...",he was a famous sports announcer and this reci...,"['butter', 'sugar', 'eggs', 'sour cream', 'bak...",9
37392,cashew gravy vegetarian vegan,265342,60,424741,2007-11-13,"['60-minutes-or-less', 'time-to-make', 'course...","[152.4, 18.0, 6.0, 13.0, 8.0, 11.0, 3.0]",17,['saute onion in oil until soft and slightly b...,i make this to go with the lentil and wild ric...,"['oil', 'onion', 'cashews', 'garlic cloves', '...",7


In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 231637 entries, 0 to 231636
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   name            231636 non-null  str  
 1   id              231637 non-null  int64
 2   minutes         231637 non-null  int64
 3   contributor_id  231637 non-null  int64
 4   submitted       231637 non-null  str  
 5   tags            231637 non-null  str  
 6   nutrition       231637 non-null  str  
 7   n_steps         231637 non-null  int64
 8   steps           231637 non-null  str  
 9   description     226658 non-null  str  
 10  ingredients     231637 non-null  str  
 11  n_ingredients   231637 non-null  int64
dtypes: int64(5), str(7)
memory usage: 21.2 MB


In [36]:
df.isna().sum()

name                 1
id                   0
minutes              0
contributor_id       0
submitted            0
tags                 0
nutrition            0
n_steps              0
steps                0
description       4979
ingredients          0
n_ingredients        0
dtype: int64

In [37]:
# Drop rows missing critical information
df_clean = df.dropna(subset=['name', 'ingredients', 'steps', 'minutes']).copy()

In [38]:
# Filter out ridiculous outliers (e.g., prep time > 5 hours or 0 minutes)
df_clean = df_clean[(df_clean['minutes'] > 0) & (df_clean['minutes'] <= 300)]

In [39]:
# 3. Filter out recipes with too few ingredients (e.g., "water") 
df_clean = df_clean[df_clean['n_ingredients'] >= 3]

In [40]:
print(f"Original recipe count: {len(df)}")
print(f"Cleaned recipe count: {len(df_clean)}")

Original recipe count: 231637
Cleaned recipe count: 218569


In [41]:
df_clean['ingredients'].sample(5)

76982     ['chicken quarters', 'lemongrass', 'purple oni...
174217    ['jumbo pasta shells', 'skinless chicken breas...
145093    ['flour', 'salt', 'pepper', 'fillets', 'vegeta...
61803     ['olive oil', 'onions', 'carrot', 'celery', 'b...
64126     ['chicken thighs', 'all-purpose flour', 'chick...
Name: ingredients, dtype: str

In [42]:
def clean_list_string(text):
    """Converts "['item1', 'item2']" into "item1, item2" """
    try:
        # Safely evaluate the string as a Python list
        actual_list = ast.literal_eval(text)
        return ", ".join(actual_list)
    except:
        return text

In [43]:
# Apply the cleaning function to ingredients
print("Cleaning ingredients...")
df_clean['clean_ingredients'] = df_clean['ingredients'].apply(clean_list_string)

Cleaning ingredients...


In [44]:
# Apply a slightly different format for steps so they are numbered
def format_steps(text):
    try:
        step_list = ast.literal_eval(text)
        return "\n".join([f"Step {i+1}: {step}" for i, step in enumerate(step_list)])
    except:
        return text

In [45]:
print("Cleaning steps...")
df_clean['clean_steps'] = df_clean['steps'].apply(format_steps)

Cleaning steps...


In [46]:
# Verify the cleanup
print("\nExample of cleaned ingredients:\n", df_clean['clean_ingredients'].iloc[0])
print("\nExample of cleaned steps:\n", df_clean['clean_steps'].iloc[0])


Example of cleaned ingredients:
 winter squash, mexican seasoning, mixed spice, honey, butter, olive oil, salt

Example of cleaned steps:
 Step 1: make a choice and proceed with recipe
Step 2: depending on size of squash , cut into half or fourths
Step 3: remove seeds
Step 4: for spicy squash , drizzle olive oil or melted butter over each cut squash piece
Step 5: season with mexican seasoning mix ii
Step 6: for sweet squash , drizzle melted honey , butter , grated piloncillo over each cut squash piece
Step 7: season with sweet mexican spice mix
Step 8: bake at 350 degrees , again depending on size , for 40 minutes up to an hour , until a fork can easily pierce the skin
Step 9: be careful not to burn the squash especially if you opt to use sugar or butter
Step 10: if you feel more comfortable , cover the squash with aluminum foil the first half hour , give or take , of baking
Step 11: if desired , season with salt


In [47]:
# Select only the columns we actually need for the RAG pipeline
final_columns = ['id', 'name', 'minutes', 'clean_ingredients', 'clean_steps']
df_final = df_clean[final_columns]

In [48]:
# Quality Control: Filter out overly simple/junk recipes
print(f"Original count: {len(df_clean)}")
df_optimized = df_clean[(df_clean['n_ingredients'] >= 3) & (df_clean['n_steps'] >= 2)].copy()

Original count: 218569


In [49]:
# Drop duplicate recipe names (keep the first one)
df_optimized = df_optimized.drop_duplicates(subset=['name'])
print(f"Count after quality filtering: {len(df_optimized)}")

Count after quality filtering: 215044


In [50]:
# Token/Length Analysis: Create a word count column for the steps
df_optimized['step_word_count'] = df_optimized['clean_steps'].apply(lambda x: len(str(x).split()))

In [51]:
# Filter out recipes that are too long (e.g., over 300 words for just instructions)
df_optimized = df_optimized[df_optimized['step_word_count'] < 300]

In [52]:
# 3. The "Rich Context" Strategy: Create the exact string we will send to the embedding model
def create_embedding_text(row):
    return f"search_document: Recipe for {row['name']}. Ready in {row['minutes']} minutes. Ingredients: {row['clean_ingredients']}."

df_optimized['text_to_embed'] = df_optimized.apply(create_embedding_text, axis=1)

In [53]:
print("\nSample string ready for the embedding model:")
print(df_optimized['text_to_embed'].iloc[0])


Sample string ready for the embedding model:
search_document: Recipe for arriba   baked winter squash mexican style. Ready in 55 minutes. Ingredients: winter squash, mexican seasoning, mixed spice, honey, butter, olive oil, salt.


In [54]:
# Save to a new file
df_optimized.to_csv('./data/optimized_recipes_for_rag.csv', index=False)
print("Saved successfully!")

Saved successfully!


In [55]:
df_optimized.shape

(207268, 16)

In [56]:
df_optimized.sample(2)

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients,clean_ingredients,clean_steps,step_word_count,text_to_embed
154340,pasto allo zafferano swiss noodles with saffron,456011,20,1058097,2011-05-13,"['30-minutes-or-less', 'time-to-make', 'course...","[578.7, 48.0, 6.0, 24.0, 42.0, 80.0, 17.0]",14,['cook pork in a large skillet over medium-hig...,"this recipe is from the canton of ticino, the ...","['ground pork', 'salt', 'white pepper', 'dried...",12,"ground pork, salt, white pepper, dried basil, ...",Step 1: cook pork in a large skillet over medi...,152,search_document: Recipe for pasto allo zaffera...
227306,wild addicting dip,123835,15,217657,2005-05-28,"['15-minutes-or-less', 'time-to-make', 'course...","[225.5, 29.0, 7.0, 14.0, 14.0, 38.0, 2.0]",3,"['spread each layer evenly in order given', 'c...","wow, the name says it all. we had a huge buffe...","['bean dip', 'lemon juice', 'salt', 'pepper', ...",11,"bean dip, lemon juice, salt, pepper, avocados,...",Step 1: spread each layer evenly in order give...,22,search_document: Recipe for wild addicting dip...
